# Heady™ Monte Carlo Simulation Engine
## Python SDK — Interactive Notebook

© 2024-2026 HeadySystems Inc. All Rights Reserved.

This notebook demonstrates the Monte Carlo simulation engine for:
- **Operational readiness scoring** from system signals
- **Full probabilistic simulation** with risk factors and mitigations
- **Reproducible PRNG** using the Mulberry32 algorithm
- **Risk grading** with Wilson confidence intervals

---

## Setup

Install dependencies and import the SDK.

In [ ]:
# Install dependencies (run once in Colab)
!pip install numpy matplotlib plotly -q

# If running from the SDK directory, add to path
import sys
sys.path.insert(0, '..')
sys.path.insert(0, '../python')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from core.monte_carlo import (
    MonteCarloEngine,
    mulberry32,
    RiskGrade,
    RiskFactor,
    ReadinessSignals,
    score_to_grade,
)

print('Heady™ Monte Carlo Engine loaded successfully')

---
## 1. Mulberry32 PRNG — Reproducible Randomness

The engine uses the Mulberry32 seeded PRNG for reproducible simulations.
Same seed always produces the same sequence.

In [ ]:
# Create two PRNGs with the same seed — they produce identical sequences
rand_a = mulberry32(42)
rand_b = mulberry32(42)

print('Same seed = identical sequences:')
for i in range(5):
    a, b = rand_a(), rand_b()
    print(f'  Step {i}: {a:.6f} == {b:.6f} → {a == b}')

# Visualize the distribution
rand = mulberry32(42)
samples = [rand() for _ in range(50000)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(samples, bins=100, color='#6366f1', alpha=0.8, edgecolor='white')
axes[0].set_title('Mulberry32 Distribution (50K samples)', fontsize=14)
axes[0].set_xlabel('Value')
axes[0].set_ylabel('Count')

# Autocorrelation check
axes[1].scatter(samples[:-1], samples[1:], alpha=0.02, s=1, color='#6366f1')
axes[1].set_title('Autocorrelation (consecutive pairs)', fontsize=14)
axes[1].set_xlabel('Sample n')
axes[1].set_ylabel('Sample n+1')
plt.tight_layout()
plt.show()
print(f'Mean: {np.mean(samples):.4f} (expected: 0.5000)')
print(f'Std:  {np.std(samples):.4f} (expected: ~0.2887)')

---
## 2. Quick Readiness Assessment

Instant operational readiness score from system signals.
Each signal is weighted and combined into a 0-100 score with a risk grade.

In [ ]:
engine = MonteCarloEngine()

# Scenario 1: Perfect system
perfect = engine.quick_readiness(ReadinessSignals())
print(f'Perfect system: score={perfect.score}, grade={perfect.grade.value}')

# Scenario 2: Stressed system
stressed = engine.quick_readiness(ReadinessSignals(
    error_rate=0.15,
    cpu_pressure=0.7,
    memory_pressure=0.6,
    service_health_ratio=0.8,
    open_incidents=2,
))
print(f'Stressed system: score={stressed.score}, grade={stressed.grade.value}')

# Scenario 3: Critical system
critical = engine.quick_readiness(ReadinessSignals(
    error_rate=0.5,
    last_deploy_success=False,
    cpu_pressure=0.95,
    memory_pressure=0.9,
    service_health_ratio=0.3,
    open_incidents=8,
))
print(f'Critical system: score={critical.score}, grade={critical.grade.value}')

In [ ]:
# Visualize the breakdown
scenarios = {
    'Perfect': perfect,
    'Stressed': stressed,
    'Critical': critical,
}

grade_colors = {
    'GREEN': '#22c55e', 'YELLOW': '#eab308',
    'ORANGE': '#f97316', 'RED': '#ef4444',
}

fig = make_subplots(rows=1, cols=3, subplot_titles=list(scenarios.keys()),
                     specs=[[{'type': 'domain'}]*3])

for i, (name, result) in enumerate(scenarios.items(), 1):
    color = grade_colors[result.grade.value]
    fig.add_trace(
        go.Indicator(
            mode='gauge+number',
            value=result.score,
            title={'text': f'{result.grade.value}'},
            gauge={
                'axis': {'range': [0, 100]},
                'bar': {'color': color},
                'steps': [
                    {'range': [0, 40], 'color': '#fee2e2'},
                    {'range': [40, 60], 'color': '#ffedd5'},
                    {'range': [60, 80], 'color': '#fef9c3'},
                    {'range': [80, 100], 'color': '#dcfce7'},
                ],
            },
        ),
        row=1, col=i
    )

fig.update_layout(height=350, title_text='Heady™ Operational Readiness Dashboard')
fig.show()

---
## 3. Full Monte Carlo Simulation

Run thousands of probabilistic iterations to assess deployment risk.

In [ ]:
# Define risk factors for a production deployment
deployment_risks = [
    RiskFactor(name='Network Partition', probability=0.08, impact=0.7, mitigation='Multi-region failover'),
    RiskFactor(name='Database Overload', probability=0.12, impact=0.6, mitigation='Connection pooling'),
    RiskFactor(name='Memory Leak', probability=0.05, impact=0.5, mitigation='Auto-restart watchdog'),
    RiskFactor(name='API Rate Limit', probability=0.15, impact=0.3),
    RiskFactor(name='Config Drift', probability=0.10, impact=0.4, mitigation='GitOps sync'),
    RiskFactor(name='SSL Expiry', probability=0.02, impact=0.9, mitigation='Auto-renewal'),
    RiskFactor(name='DNS Propagation', probability=0.06, impact=0.4),
]

result = engine.run_full_cycle(
    name='Production Deployment v3.1.0',
    seed=42,
    risk_factors=deployment_risks,
    iterations=50000,
)

print(f'Scenario:        {result.scenario}')
print(f'Confidence:      {result.confidence}%')
print(f'Risk Grade:      {result.risk_grade.value}')
print(f'Failure Rate:    {result.failure_rate:.4f}')
print(f'Outcomes:')
print(f'  Success:       {result.outcomes.success:,}')
print(f'  Partial:       {result.outcomes.partial:,}')
print(f'  Failure:       {result.outcomes.failure:,}')
print(f'95% CI:          [{result.confidence_bounds.lower:.4f}, {result.confidence_bounds.upper:.4f}]')
print(f'Top Mitigations: {result.top_mitigations}')

In [ ]:
# Visualize outcomes
labels = ['Success', 'Partial', 'Failure']
values = [result.outcomes.success, result.outcomes.partial, result.outcomes.failure]
colors = ['#22c55e', '#eab308', '#ef4444']

fig = make_subplots(rows=1, cols=2, specs=[[{'type': 'domain'}, {'type': 'xy'}]],
                     subplot_titles=['Outcome Distribution', 'Convergence Analysis'])

fig.add_trace(go.Pie(labels=labels, values=values, marker=dict(colors=colors),
                      hole=0.4, textinfo='label+percent'), row=1, col=1)

# Convergence: run with increasing iterations
iter_counts = [100, 500, 1000, 2500, 5000, 10000, 25000, 50000]
confidences = []
for n in iter_counts:
    r = engine.run_full_cycle(name='convergence', seed=42, risk_factors=deployment_risks, iterations=n)
    confidences.append(r.confidence)

fig.add_trace(go.Scatter(x=iter_counts, y=confidences, mode='lines+markers',
                          name='Confidence', line=dict(color='#6366f1', width=3),
                          marker=dict(size=8)), row=1, col=2)
fig.update_xaxes(title_text='Iterations', type='log', row=1, col=2)
fig.update_yaxes(title_text='Confidence %', row=1, col=2)
fig.update_layout(height=450, title_text=f'Monte Carlo: {result.scenario}')
fig.show()

---
## 4. Comparative Risk Analysis

Compare multiple deployment strategies with different mitigation profiles.

In [ ]:
# Strategy A: Minimal mitigations
risks_minimal = [
    RiskFactor(name='Network', probability=0.15, impact=0.7),
    RiskFactor(name='Database', probability=0.12, impact=0.6),
    RiskFactor(name='Memory', probability=0.10, impact=0.5),
    RiskFactor(name='Config', probability=0.08, impact=0.4),
]

# Strategy B: Partial mitigations
risks_partial = [
    RiskFactor(name='Network', probability=0.15, impact=0.7, mitigation='Failover'),
    RiskFactor(name='Database', probability=0.12, impact=0.6),
    RiskFactor(name='Memory', probability=0.10, impact=0.5, mitigation='Watchdog'),
    RiskFactor(name='Config', probability=0.08, impact=0.4),
]

# Strategy C: Full mitigations
risks_full = [
    RiskFactor(name='Network', probability=0.15, impact=0.7, mitigation='Multi-region'),
    RiskFactor(name='Database', probability=0.12, impact=0.6, mitigation='Pooling'),
    RiskFactor(name='Memory', probability=0.10, impact=0.5, mitigation='Watchdog'),
    RiskFactor(name='Config', probability=0.08, impact=0.4, mitigation='GitOps'),
]

strategies = {
    'Minimal': risks_minimal,
    'Partial': risks_partial,
    'Full': risks_full,
}

results = {}
for name, factors in strategies.items():
    r = engine.run_full_cycle(name=name, seed=42, risk_factors=factors, iterations=25000)
    results[name] = r
    print(f'{name:10s} → Confidence: {r.confidence:3d}%  Grade: {r.risk_grade.value:6s}  '
          f'Failures: {r.outcomes.failure:,}')

# Bar chart comparison
fig = go.Figure()
for name, r in results.items():
    fig.add_trace(go.Bar(
        name=name,
        x=['Success', 'Partial', 'Failure'],
        y=[r.outcomes.success, r.outcomes.partial, r.outcomes.failure],
    ))
fig.update_layout(barmode='group', title='Strategy Comparison (25K iterations)',
                   yaxis_title='Count', height=450)
fig.show()

---
## 5. Engine Status & History

Track all simulation runs and engine health.

In [ ]:
status = engine.status()
print(f'Total runs: {status["total_runs"]}')
print(f'Last run:   {status["last_run"]}')

print('\nRecent history:')
for entry in engine.get_history(limit=5):
    r = entry.result
    print(f'  [{r.risk_grade.value:6s}] {r.scenario:30s} → {r.confidence}% confidence ({r.iterations:,} iterations)')